# 9.5 — Windows over Event Time

**Chapter 9, §9.4.2 and §9.4.3.** *Which interval does a count belong to, and when
does the engine stop waiting for stragglers?*

A total over an unbounded stream is never finished, so the answerable question is
always "how many in this interval of event time". This notebook puts a handful of
events on a timeline, counts them with tumbling and then with sliding windows, and
finally watches the watermark refuse a record that arrived too late to matter.

Event time is the clock at the source, carried inside the record. Processing time
is Spark's own clock. Nothing here ever groups by the second one. Every input is
written out in full below, so you can work out each answer before running the cell.

In [1]:
import os, shutil, tempfile, time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max as smax, window

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch09-windows")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

spark = (
    SparkSession.builder.appName("CS777-9.5")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    # Windows are printed back as timestamps, so fix the session zone and the
    # printed boundaries line up with the event times written below.
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.2.0'

## The input

Five events, one word each, on the morning of 1 January 2026. Written to a CSV
file that a streaming query will pick up.

In [2]:
DAY = "2026-01-01"
FIRST = [("12:02", "dog"), ("12:04", "cat"), ("12:07", "dog"),
         ("12:09", "cat"), ("12:14", "dog")]

def write_batch(directory, name, rows, order):
    os.makedirs(directory, exist_ok=True)
    path = os.path.join(directory, name)
    with open(path, "w") as fh:
        for clock, word in rows:
            fh.write(f"{DAY} {clock}:00,{word}\n")
    # The file source consumes files in modification-time order, so the arrival
    # order has to be stamped rather than assumed. Files written in the same
    # second would otherwise arrive in no particular order.
    stamp = time.time() - 3600 + 10 * order
    os.utime(path, (stamp, stamp))

print(", ".join(f"({c}, {w})" for c, w in FIRST))

(12:02, dog), (12:04, cat), (12:07, dog), (12:09, cat), (12:14, dog)


## Reading a directory as a stream

The schema is declared rather than inferred: a streaming query has to know the
shape of a file before that file exists. `maxFilesPerTrigger` set to one makes
every file its own micro-batch, so arrivals stay separate and observable.

In [3]:
SCHEMA = "event_time TIMESTAMP, word STRING"

def read_dir(directory):
    return (
        spark.readStream.schema(SCHEMA)
        .option("maxFilesPerTrigger", 1)
        .csv(directory)
    )

# Run a query over whatever is already on disk, stop when it runs out, and hand
# back the result table.
def run_to_memory(frame, name, mode="complete"):
    query = (
        frame.writeStream.queryName(name)
        .outputMode(mode)
        .format("memory")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
    return query

## Tumbling windows: every record in exactly one

Ten minutes long, no overlap. Four of the five events fall in 12:00–12:10 and one
in 12:10–12:20, and nothing is counted twice.

In [4]:
TUMBLING_DIR = os.path.join(SCRATCH, "tumbling")
write_batch(TUMBLING_DIR, "batch1.csv", FIRST, order=0)

tumbling = (
    read_dir(TUMBLING_DIR)
    .groupBy(window(col("event_time"), "10 minutes"), col("word"))
    .count()
)
run_to_memory(tumbling, "tumbling")
tumbling_result = spark.table("tumbling").orderBy("window", "word")
tumbling_result.show(truncate=False)

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|cat |2    |
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|dog |2    |
|{2026-01-01 12:10:00, 2026-01-01 12:20:00}|dog |1    |
+------------------------------------------+----+-----+



## Sliding windows: the same records, twice the state

Still ten minutes long, but a new window opens every five. Each record now belongs
to two of them, so the result table doubles for exactly the same input. The factor
is the window length divided by the slide.

In [5]:
SLIDING_DIR = os.path.join(SCRATCH, "sliding")
write_batch(SLIDING_DIR, "batch1.csv", FIRST, order=0)

sliding = (
    read_dir(SLIDING_DIR)
    .groupBy(window(col("event_time"), "10 minutes", "5 minutes"), col("word"))
    .count()
)
run_to_memory(sliding, "sliding")
sliding_result = spark.table("sliding").orderBy("window", "word")
sliding_result.show(truncate=False)

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2026-01-01 11:55:00, 2026-01-01 12:05:00}|cat |1    |
|{2026-01-01 11:55:00, 2026-01-01 12:05:00}|dog |1    |
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|cat |2    |
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|dog |2    |
|{2026-01-01 12:05:00, 2026-01-01 12:15:00}|cat |1    |
|{2026-01-01 12:05:00, 2026-01-01 12:15:00}|dog |2    |
|{2026-01-01 12:10:00, 2026-01-01 12:20:00}|dog |1    |
+------------------------------------------+----+-----+



In [6]:
# The same five records, counted twice as often. An assertion that runs, rather
# than a sentence claiming the arithmetic.
tumbling_total = tumbling_result.selectExpr("sum(count) AS t").first()["t"]
sliding_total = sliding_result.selectExpr("sum(count) AS t").first()["t"]
print("records counted, tumbling:", tumbling_total)
print("records counted, sliding: ", sliding_total)
assert sliding_total == 2 * tumbling_total

records counted, tumbling: 5
records counted, sliding:  10


## The watermark

A window cannot stay open forever, so the engine tracks the largest event time it
has seen and subtracts a lateness allowance. That moving threshold is the
watermark: once it passes the end of a window, the window is finalized and its
state is thrown away, and a record for it arriving afterwards has nowhere to go.

Two details of the mechanism matter, and both are visible below.

* **The threshold lags behind the arrivals.** The watermark a batch is judged
  against was computed from data that arrived before it, so a record is never
  refused by a threshold its own batch has just raised.
* **The test is on the window, not on the record.** A record is refused once the
  window it belongs to has been finalized and its state discarded.

Four files arrive, one per micro-batch, with a ten-minute allowance. The right-hand
column is the watermark each batch reports, which is the largest event time seen
before it, less the allowance:

| batch | rows | watermark reported |
|---|---|---|
| 0 | the five events above | none yet |
| 1 | (12:09, cat), (12:21, owl) | 12:04 |
| 2 | (12:23, owl) | 12:11 |
| 3 | (12:04, donkey) | 12:13 |

`(12:09, cat)` is out of order, but it arrives while 12:00–12:10 is still open and
is counted, taking that window's total for `cat` to three. `(12:04, donkey)`
arrives in batch 3, by which point the watermark has long passed 12:10 and the
window it needs is gone. It is refused, and the engine reports the refusal itself
rather than leaving you to infer it from a missing row.

In [7]:
WM_DIR = os.path.join(SCRATCH, "watermark")
write_batch(WM_DIR, "1.csv", FIRST, order=0)
write_batch(WM_DIR, "2.csv", [("12:09", "cat"), ("12:21", "owl")], order=1)
write_batch(WM_DIR, "3.csv", [("12:23", "owl")], order=2)
write_batch(WM_DIR, "4.csv", [("12:04", "donkey")], order=3)

watermarked = (
    read_dir(WM_DIR)
    .withWatermark("event_time", "10 minutes")
    .groupBy(window(col("event_time"), "10 minutes"), col("word"))
    .count()
)
# Update mode: each trigger emits only the windows whose counts changed. Complete
# mode would keep every window alive forever and there would be nothing to watch.
query = run_to_memory(watermarked, "watermarked", mode="update")
history = list(query.recentProgress)
print(f"{len(history)} micro-batches")

4 micro-batches


In [8]:
# What the engine did with each batch: rows read, rows refused as too late,
# windows still held open, and the watermark it reports.
for progress in history:
    state = progress["stateOperators"][0]
    watermark = progress["eventTime"].get("watermark", "-")
    print(
        f"batch {progress['batchId']}: "
        f"read {progress['numInputRows']}, "
        f"refused as late {state['numRowsDroppedByWatermark']}, "
        f"windows held {state['numRowsTotal']}, "
        f"watermark reported {watermark[11:16]}"
    )

batch 0: read 5, refused as late 0, windows held 3, watermark reported 00:00
batch 1: read 2, refused as late 0, windows held 4, watermark reported 12:04
batch 2: read 1, refused as late 0, windows held 2, watermark reported 12:11
batch 3: read 1, refused as late 1, windows held 2, watermark reported 12:13


In [9]:
# Update mode emits a row every time a count changes, so the memory table holds
# several versions of the same window. The latest value for each is the answer.
latest = (
    spark.table("watermarked")
    .groupBy("window", "word")
    .agg(smax("count").alias("count"))
    .orderBy("window", "word")
)
latest.show(truncate=False)

print("windows containing donkey:", latest.where(col("word") == "donkey").count())
print("count for cat in 12:00-12:10:",
      latest.where((col("word") == "cat") & (col("window.start") == f"{DAY} 12:00:00"))
            .first()["count"])

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|cat |3    |
|{2026-01-01 12:00:00, 2026-01-01 12:10:00}|dog |2    |
|{2026-01-01 12:10:00, 2026-01-01 12:20:00}|dog |1    |
|{2026-01-01 12:20:00, 2026-01-01 12:30:00}|owl |2    |
+------------------------------------------+----+-----+

windows containing donkey: 0
count for cat in 12:00-12:10: 3


In [10]:
# cat arrived out of order and was still counted, three times in all: 12:04,
# 12:09, and the second 12:09 that arrived a batch later. donkey arrived after
# its window was finalized and appears nowhere.
assert latest.where(col("word") == "donkey").count() == 0
assert sum(p["stateOperators"][0]["numRowsDroppedByWatermark"] for p in history) == 1
print("as expected: one record refused, and it is not in any window")

as expected: one record refused, and it is not in any window


## What this shows

A window is a grouping column derived from event time, which is why an ordinary
`groupBy` expresses it and why nothing about the surrounding query has to change.
Overlapping windows buy smoother and more frequent aggregates, and pay for them in
state at the rate of the length divided by the slide: ten minutes sliding every
minute costs ten times the state of the tumbling equivalent.

The watermark is what makes a query survivable over weeks rather than minutes. It
is a promise about how long the engine will keep waiting, and a record that misses
the deadline is gone from the streaming result permanently. A longer allowance
catches more stragglers and holds more state; a shorter one holds less and loses
more. When a streaming total looks too small, this is the first setting to examine,
and a periodic batch job over the retained history is the usual way of recovering
what the stream refused.

In [11]:
spark.stop()